### Miguel Baños Baladrón
### Miguel Pérez Francos
### Rodrigo Touceda Tapias

# Práctica 2: Reconocimiento de Actividades Humanas (HAR)
## Preparación y Carga de datos (20%)

**Objetivo:** Cargar, unificar y preparar el conjunto de datos para el entrenamiento de modelos de clasificación.

**Estructura del Dataset:**
* **Origen:** Datos de sensores inerciales (acelerómetro y giroscopio).
* **Organización:** Múltiples ficheros (uno por individuo).
* **Problema:** Clasificación multiclase (6 actividades).

# Importaciones

In [1]:
include("setup.jl");
include("helpers.jl");

┌ Info: lib_lightgbm found in system dirs!
└ @ LightGBM C:\Users\Pc\.julia\packages\LightGBM\Ysr56\src\LightGBM.jl:37


Entorno y modelos cargados correctamente


## 1. Carga y unificación de los datos

In [2]:
# Carga de archivos CSV 
csv_inv_a = glob("Investigador A/day */*.csv", DATA_PATH)
csv_inv_b = glob("Investigador B/*.csv", DATA_PATH)

# Concatenación vertical de todos los ficheros
all_csv = vcat(csv_inv_a, csv_inv_b)
dfs = [CSV.read(file, DataFrame) for file in all_csv]
df_total = vcat(dfs...)

# Información sobre el dataset
println("=== Resumen del Dataset ===")
println("Número de Instancias (Filas):  ", nrow(df_total))
println("Número de Variables (Cols):    ", ncol(df_total))
println("Número de Individuos:          ", length(unique(df_total.subject)))
println("Número de Clases de Actividad: ", length(unique(df_total.Activity)))

=== Resumen del Dataset ===
Número de Instancias (Filas):  10299
Número de Variables (Cols):    563
Número de Individuos:          30
Número de Clases de Actividad: 6


## 2. Análisis de valores ausentes
- Se analiza la presencia de valores nulos (NaN/Missing) en el conjunto de datos. Siguiendo las indicaciones del estudio, asumimos un mecanismo **MCAR (Missing Completely At Random)**.
- Para analizar la calidad del dataset, se evaluó la presencia de valores ausentes tanto a nivel global como por variable. El porcentaje total de celdas con valores nulos en el dataset (`df_total`) es cercano al **1%** A nivel de columnas, **175 de las 563 variables** presentan al menos un valor faltante. Las variables con mayor proporción de valores ausentes alcanzan ligeramente más del **10%**. -->

In [3]:
function nulls_analysis(df::DataFrame; top=10)

    # Porcentaje total de nulos
    total_missing = sum(count(ismissing, df[!, col]) for col in names(df))
    total_values  = nrow(df) * ncol(df)
    pct_total = (total_missing / total_values) * 100
    println("Porcentaje de valores ausentes en el dataset: $(pct_total) %")

    # Porcentaje de nulos por columna
    df_nulls = DataFrame(
        Variable = names(df),
        NullsPercentage = [mean(ismissing.(df[!, c])) * 100 for c in names(df)]
    )

    # Variables con algún valor nulo
    df_nulls_pos = filter(:NullsPercentage => p -> p > 0, df_nulls)
    println("Variables con algún valor ausente: ", nrow(df_nulls_pos))

    # Ordenar de mayor a menor
    sort!(df_nulls_pos, :NullsPercentage, rev=true)

    # Mostrar top variables con más nulos
    if nrow(df_nulls_pos) > 0
        n_show = min(top, nrow(df_nulls_pos))
        println("\nTop $n_show variables con valores ausentes:")
        pretty_table(first(df_nulls_pos, n_show))
    else
        println("Ninguna columna contiene valores ausentes.")
    end

    return df_nulls_pos, pct_total
end

nulls_analysis(df_total);

Porcentaje de valores ausentes en el dataset: 0.9984242033534787 %
Variables con algún valor ausente: 175

Top 10 variables con valores ausentes:
┌──────────────────────────┬─────────────────┐
│                 Variable │ NullsPercentage │
│                   String │         Float64 │
├──────────────────────────┼─────────────────┤
│       tBodyGyroMag-mad() │         10.0301 │
│       tBodyGyroMag-iqr() │         10.0301 │
│         fBodyAcc-mad()-Y │         10.0204 │
│    fBodyAccJerk-mean()-X │         10.0204 │
│  tBodyAccJerk-energy()-X │          10.001 │
│ tBodyAccJerk-entropy()-Y │          10.001 │
│        tBodyAccMag-max() │          10.001 │
│     tGravityAccMag-std() │          10.001 │
│ tGravityAccMag-entropy() │          10.001 │
│         fBodyAcc-std()-X │          10.001 │
└──────────────────────────┴─────────────────┘


## 3. Tratamiento y transformación de datos
- En esta sección preparamos el conjunto de datos para su uso en los algoritmos de clasificación. El objetivo es obtener un dataset completamente limpio, sin valores ausentes y con la variable objetivo correctamente codificada. Como los datos proceden de sensores biométricos de distintos individuos, imputaremos los datos ausentes con la mediana por sujeto, por su robustez frente a outliers y para no introducir sesgos globales.

In [4]:
df_imputed = deepcopy(df_total)

function impute_feature(df::DataFrame)
    individuals = groupby(df, :subject)

    for individual in individuals
        for col in names(individual)

            if col in (:subject, :Activity)
                continue
            end
        
            col_data = individual[!, col]

            if eltype(skipmissing(col_data)) <: Number #  Solo imputamos variables numéricas
                med = median(skipmissing(col_data))
                replace!(col_data, missing => med)
            end
        end
    end

    return df
end

impute_feature(df_imputed)
nulls_analysis(df_imputed); # Verificamos que ya no hay nulos

# La etiqueta debe ser categórica
df_imputed.Activity = categorical(df_imputed.Activity)

# Separamos features y target (características y etiqueta)
y = df_imputed.Activity
x = DataFrames.select(df_imputed, Not([:subject, :Activity]))

# Mostramos que estos tratamientos se aplican correctamente
println("\nTipo de la variable objetivo (y): $(eltype(df_imputed.Activity))")
println("Número de features (columnas en dataset de features): ", ncol(x))

Porcentaje de valores ausentes en el dataset: 0.0 %
Variables con algún valor ausente: 0
Ninguna columna contiene valores ausentes.

Tipo de la variable objetivo (y): CategoricalArrays.CategoricalValue{String31, UInt32}
Número de features (columnas en dataset de features): 561


## 4. Partición Holdout
Reservamos un **10% de los individuos** para el conjunto de prueba final, asegurando que ningún dato de estos sujetos se utilice durante el entrenamiento o la validación cruzada.

In [5]:
# Obtener lista de sujetos únicos
subjects = unique(df_imputed.subject)
Random.seed!(SEED)
shuffle!(subjects)

# Cálculo de índices de corte (10% Test)
n_test = round(Int, length(subjects) * 0.10)
test_subjects = subjects[1:n_test]
trainval_subjects = subjects[n_test+1:end]

println("=== Partición Hold-Out (Subject-wise) ===")
println("Total Sujetos: $(length(subjects))")
println("Sujetos en Train/Val (90%): $(length(trainval_subjects))")
println("Sujetos en Test (10%):      $(length(test_subjects))")
println("IDs de Sujetos Train+Val:   $trainval_subjects")
println("IDs de Sujetos Test:        $test_subjects")

# Filtrado de DataFrames
df_trainval = filter(row -> row.subject in trainval_subjects, df_imputed)
df_test     = filter(row -> row.subject in test_subjects,     df_imputed)

# Preparación de matrices X e y finales
X_trainval = DataFrames.select(df_trainval, Not([:subject, :Activity]))
y_trainval = df_trainval.Activity

# Generación de Folds para CV (asegurando independencia de sujetos)
folds_trainval = subject_folds(df_trainval, k=5, seed=SEED)

# Guardado
@save "datos_procesados.jld2" df_trainval df_test X_trainval y_trainval folds_trainval
println("\nDatos procesados guardados exitosamente.")

=== Partición Hold-Out (Subject-wise) ===
Total Sujetos: 30
Sujetos en Train/Val (90%): 27
Sujetos en Test (10%):      3
IDs de Sujetos Train+Val:   [20, 27, 30, 19, 29, 7, 9, 23, 17, 8, 4, 24, 10, 2, 3, 13, 26, 11, 28, 6, 12, 5, 16, 21, 15, 1, 14]
IDs de Sujetos Test:        [25, 18, 22]

Datos procesados guardados exitosamente.


## 5. Comprobación del balance de clases
- Se examina el balance de clases para conocer cómo se distribuyen las muestras entre las distintas categorías del conjunto de datos. Un conjunto de datos desbalanceado puede provocar que el modelo no aprenda correctamente. Como vemos en los porcentajes, tenemos un conjunto de datos balanceado.

In [8]:
function check_class_balance(y, dataset_name)
    counts = countmap(y)
    total = length(y)
    
    println("\nDistribución de clases en: $dataset_name (Total: $total)")
    
    # Convertimos a DataFrame para visualización limpia
    df_balance = DataFrame(
        Clase = collect(keys(counts)),
        N = collect(values(counts)),
        Porcentaje = [round(v/total*100, digits=2) for v in values(counts)]
    )
    sort!(df_balance, :Porcentaje, rev=true)
    
    pretty_table(df_balance)
end

check_class_balance(y_trainval, "Train/Validation")
check_class_balance(df_test.Activity, "Test (Holdout)")


Distribución de clases en: Train/Validation (Total: 9205)
┌──────────────────────────────────────────────────────┬───────┬────────────┐
│                                                Clase │     N │ Porcentaje │
│ CategoricalArrays.CategoricalValue{String31, UInt32} │ Int64 │    Float64 │
├──────────────────────────────────────────────────────┼───────┼────────────┤
│                                               LAYING │  1734 │      18.84 │
│                                             STANDING │  1696 │      18.42 │
│                                              SITTING │  1593 │      17.31 │
│                                              WALKING │  1546 │       16.8 │
│                                     WALKING_UPSTAIRS │  1379 │      14.98 │
│                                   WALKING_DOWNSTAIRS │  1257 │      13.66 │
└──────────────────────────────────────────────────────┴───────┴────────────┘

Distribución de clases en: Test (Holdout) (Total: 1094)
┌─────────────────────────